# 3 - Multiple defects, multiple combined sources, different wavelengths

The harder PhD variant: a plate with 3 defects, probed by 4 sources fired together (each a different frequency / wavelength and delay) superimposed in one wavefield. The adjoint machinery is unchanged and recovers all defects.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/03_multidefect_multisource.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    # Put the package on the path directly. An editable `pip install -e .` of a
    # src-layout package is not picked up by the already-running Colab kernel, and
    # torch/numpy/matplotlib are pre-installed on Colab, so no pip step is needed.
    sys.path.insert(0, os.path.abspath("src"))

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)
print("Note: clean float64 needs CPU or CUDA; Apple MPS runs float32 (see caveat below).")

## Combined multi-source forward on the 3-defect plate

In [ ]:
from pathlib import Path
from fwi import geometry, wavelet, plotting
from fwi.config import SimConfig, default_source_scale
from fwi.domain import build_alpha2, read_domain
from fwi.forward import forward
from fwi.adjoint import adjoint_gradient
from fwi.inversion import invert

import fwi
DATA = Path(fwi.__file__).resolve().parents[2] / 'data' / 'domain'
cfg = SimConfig(source_scale=default_source_scale(dtype))
ds = read_domain(DATA/'Domain2D_model.txt')
dt_ = read_domain(DATA/'Domain2D_Fichtner_3pt_oben.txt')
start, active = build_alpha2(ds, device=device, dtype=dtype)
true, _ = build_alpha2(dt_, device=device, dtype=dtype)

SRC_X=[117.,152.,173.,95.]; SRC_Y=[47.,31.,71.,60.]; SRC_F0=[12000.,15000.,18000.,21000.]
si, sj = geometry.make_sources(ds, SRC_X, SRC_Y)
t0=[(k+1)/f for k,f in enumerate(SRC_F0)]
sig = wavelet.gaussian_derivative(cfg, f0=SRC_F0, t0=t0, device=device, dtype=dtype)
ri, rj = geometry.make_receivers(ds, cfg)
print('wavelengths c/f0 [cm]:', [round(cfg.c_max/f*100,1) for f in SRC_F0])
with torch.no_grad():
    observed = forward(true, sig, si, sj, ri, rj, cfg).traces

## Adjoint kernel and inversion recovering the 3 defects

In [ ]:
kernel, J = adjoint_gradient(start, observed, sig, si, sj, ri, rj, cfg, active_mask=active)
display(Image(str(plotting.save_field(kernel, 'outputs/nb3_kernel.png',
    title='adjoint kernel: 3 defects, 4 combined sources'))))

alpha2_hat, history = invert(start, observed, sig, si, sj, ri, rj, cfg,
    active_mask=active, grad_fn='adjoint', n_iter=25, lr=1e4)
print(f'misfit {history[0]:.3e} -> {history[-1]:.3e} ({history[0]/history[-1]:.1f}x)')
display(Image(str(plotting.save_convergence(history, 'outputs/nb3_conv.png'))))
display(Image(str(plotting.save_field(true - start, 'outputs/nb3_true.png', title='true (3 defects)'))))
display(Image(str(plotting.save_field(alpha2_hat - start, 'outputs/nb3_update.png', title='recovered update'))))